### **Dataset Preparation**

Preserve the original datasets and create separate working copies for data cleaning.

In [3]:
import os
import zipfile
import shutil
import pandas as pd
import numpy as np

from google.colab import files

# Upload the original Olist ZIP file
uploaded = files.upload()

zip_path = next(iter(uploaded))

# Create project directories
base_path = "/content/retail_analytics"
raw_path = os.path.join(base_path, "data", "raw")
processed_path = os.path.join(base_path, "data", "processed")

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

# Extract original files into the raw folder
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(raw_path)

# Copy CSV files into the working folder
for root, dirs, filenames in os.walk(raw_path):
    for filename in filenames:
        if filename.endswith(".csv"):
            source = os.path.join(root, filename)
            destination = os.path.join(processed_path, filename)
            shutil.copy2(source, destination)

print("Raw datasets preserved.")
print("Working copies created.")
print("Number of working CSV files:",
      len([f for f in os.listdir(processed_path) if f.endswith(".csv")]))

Saving archive.zip to archive.zip
Raw datasets preserved.
Working copies created.
Number of working CSV files: 9


### **Initial Data Quality Check**

Check missing values and exact duplicate rows in all nine working datasets.

In [4]:
# Load all working CSV files
datasets = {}

for filename in os.listdir(processed_path):
    if filename.endswith(".csv"):
        table_name = filename.replace(".csv", "")
        datasets[table_name] = pd.read_csv(
            os.path.join(processed_path, filename)
        )

# Summarise missing values and exact duplicate rows
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })

quality_df = pd.DataFrame(quality_summary)

display(quality_df)

,Dataset,Rows,Missing Values,Duplicate Rows
0,product_category_name_translation,71,0,0
1,olist_order_reviews_dataset,99224,145903,0
2,olist_order_payments_dataset,103886,0,0
3,olist_products_dataset,32951,2448,0
4,olist_geolocation_dataset,1000163,0,261831
5,olist_customers_dataset,99441,0,0
6,olist_sellers_dataset,3095,0,0
7,olist_orders_dataset,99441,4908,0
8,olist_order_items_dataset,112650,0,0


### **Column-Level Missing Value Analysis**

Identify which columns contain missing values and how many are missing.

In [5]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{name}")
        display(missing.rename("Missing Count").to_frame())


olist_order_reviews_dataset


,Missing Count
review_comment_title,87656
review_comment_message,58247



olist_products_dataset


,Missing Count
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2



olist_orders_dataset


,Missing Count
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965


### **Order Timestamp Investigation**

Compare missing order timestamps with order statuses to understand why dates are unavailable.

In [6]:
orders = datasets["olist_orders_dataset"]

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in date_columns:
    print(f"\nMissing {column} by order status:")
    display(
        orders.loc[orders[column].isna(), "order_status"]
        .value_counts()
        .rename("Missing Orders")
        .to_frame()
    )


Missing order_approved_at by order status:


,Missing Orders
order_status,
canceled,141
delivered,14
created,5



Missing order_delivered_carrier_date by order status:


,Missing Orders
order_status,
unavailable,609
canceled,550
invoiced,314
processing,301
created,5
approved,2
delivered,2



Missing order_delivered_customer_date by order status:


,Missing Orders
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


### **Missing Dates in Delivered Orders**

Inspect delivered orders with missing approval or delivery timestamps.

In [7]:
delivered_date_issues = orders[
    (orders["order_status"] == "delivered") &
    (orders[date_columns].isna().any(axis=1))
]

display(
    delivered_date_issues[
        ["order_id"] + date_columns
    ]
)

,order_id,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN
5323,e04abd8149ef81b95221e88f6ed9ab6a,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN
22663,5cf925b116421afa85ee25e99b4c34fb,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23


### **Missing Timestamp Pattern Analysis**

Identify combinations of missing timestamps among delivered orders.

In [8]:
delivered = orders[orders["order_status"] == "delivered"].copy()

missing_patterns = (
    delivered[date_columns]
    .isna()
    .value_counts()
    .reset_index(name="Order Count")
)

display(missing_patterns)

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,Order Count
0,False,False,False,96455
1,True,False,False,14
2,False,False,True,7
3,False,True,False,1
4,False,True,True,1


### **Product Missing Value**

Check whether the same products have missing category and descriptive attributes.

In [9]:
products = datasets["olist_products_dataset"]

product_missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

display(
    products.loc[
        products[product_missing_columns].isna().any(axis=1),
        product_missing_columns
    ].isna().value_counts().rename("Product Count").to_frame()
)

,,,,Product Count
product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,
True,True,True,True,610


### **Product Dimension**

Inspect the two products with incomplete physical measurements.

In [10]:
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

display(
    products.loc[
        products[dimension_columns].isna().any(axis=1),
        ["product_id"] + dimension_columns
    ]
)

,product_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN


### **Product Usage**

Check whether the two products with missing measurements appear in order items.

In [11]:
order_items = datasets["olist_order_items_dataset"]

missing_dimension_ids = products.loc[
    products[dimension_columns].isna().any(axis=1),
    "product_id"
]

usage_check = order_items[
    order_items["product_id"].isin(missing_dimension_ids)
]["product_id"].value_counts()

display(
    usage_check.rename("Order Item Count").to_frame()
)

,Order Item Count
product_id,
5eb564652db742ff8f28759cd8d2652a,17
09ff539a621711667c43eba6a3bd8466,1


### **Geolocation Duplicate**

Examine how many distinct coordinate pairs exist for each ZIP-code prefix.

In [12]:
geo = datasets["olist_geolocation_dataset"]

zip_coordinate_check = (
    geo.groupby("geolocation_zip_code_prefix")
    .apply(
        lambda group: group[
            ["geolocation_lat", "geolocation_lng"]
        ].drop_duplicates().shape[0]
    )
)

print("ZIP prefixes with multiple coordinate pairs:",
      (zip_coordinate_check > 1).sum())

print("Maximum coordinate pairs for one ZIP prefix:",
      zip_coordinate_check.max())

ZIP prefixes with multiple coordinate pairs: 17781
Maximum coordinate pairs for one ZIP prefix: 746


/tmp/ipykernel_1100/1657398499.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


### **Data Cleaning**

#### **1. Remove Exact Duplicate Records**

Remove completely identical rows from the geolocation dataset while preserving records with distinct geographic coordinates.

In [13]:
# Remove exact duplicate records from geolocation data
geo = datasets["olist_geolocation_dataset"]

before = len(geo)

geo = geo.drop_duplicates().copy()

after = len(geo)

# Update the working dataset
datasets["olist_geolocation_dataset"] = geo

# Save the cleaned dataset
geo.to_csv(
    os.path.join(processed_path, "olist_geolocation_dataset.csv"),
    index=False
)

print("Rows before cleaning:", before)
print("Rows after cleaning:", after)
print("Exact duplicates removed:", before - after)

Rows before cleaning: 1000163
Rows after cleaning: 738332
Exact duplicates removed: 261831


#### **2. Handle Missing Review Comments**

Replace missing review titles and messages with empty strings while preserving all review records.

In [14]:
reviews = datasets["olist_order_reviews_dataset"]

comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

reviews[comment_columns] = reviews[comment_columns].fillna("")

datasets["olist_order_reviews_dataset"] = reviews

reviews.to_csv(
    os.path.join(processed_path, "olist_order_reviews_dataset.csv"),
    index=False
)

print("Remaining missing comment values:",
      reviews[comment_columns].isna().sum().sum())

print("Review records preserved:", len(reviews))

Remaining missing comment values: 0
Review records preserved: 99224


#### **3. Handle Missing Product Descriptive Attributes**

Preserve products with incomplete descriptive information and use an explicit placeholder for missing product categories.

In [15]:
products = datasets["olist_products_dataset"]

products["product_category_name"] = (
    products["product_category_name"].fillna("Unknown")
)

datasets["olist_products_dataset"] = products

products.to_csv(
    os.path.join(processed_path, "olist_products_dataset.csv"),
    index=False
)

print("Missing product categories:",
      products["product_category_name"].isna().sum())

print("Product records preserved:", len(products))

Missing product categories: 0
Product records preserved: 32951


#### **4. Preserve Missing Product Dimensions**

Retain products with incomplete physical measurements without introducing estimated or inaccurate values.

In [16]:
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("Products with missing dimensions:",
      products[dimension_columns].isna().any(axis=1).sum())

print("Product records preserved:", len(products))

Products with missing dimensions: 2
Product records preserved: 32951


#### **5. Preserve Missing Order Timestamps**

Retain orders with missing timestamps without assigning artificial dates or removing valid order records.

In [17]:
orders = datasets["olist_orders_dataset"]

print("Orders with missing timestamps:",
      orders[date_columns].isna().any(axis=1).sum())

print("Total order records preserved:", len(orders))

Orders with missing timestamps: 2980
Total order records preserved: 99441


#### **6. Convert Order Timestamps to Datetime**

Convert order timestamp columns into datetime format while preserving missing values.

In [18]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

datasets["olist_orders_dataset"] = orders

orders.to_csv(
    os.path.join(processed_path, "olist_orders_dataset.csv"),
    index=False
)

print("Timestamp conversion completed.")
print("Order records preserved:", len(orders))
print("Missing timestamp values:", orders[date_columns].isna().sum().sum())

Timestamp conversion completed.
Order records preserved: 99441
Missing timestamp values: 4908


#### **7. Validate Converted Order Timestamps**

Verify timestamp data types and check for invalid date values.

In [19]:
print("Order timestamp data types:")
display(orders[date_columns].dtypes.to_frame("Data Type"))

print("Invalid timestamp values:",
      orders[date_columns].isna().sum().sum())

Order timestamp data types:


,Data Type
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


Invalid timestamp values: 4908


#### **8. Save Cleaned Working Datasets**

Save the updated datasets after applying the missing-value and timestamp handling decisions.

In [20]:
for name, df in datasets.items():
    df.to_csv(
        os.path.join(processed_path, f"{name}.csv"),
        index=False
    )

print("All updated working datasets saved.")
print("Total datasets:", len(datasets))

All updated working datasets saved.
Total datasets: 9


### **Final Data Quality Validation**

Validate the cleaned datasets to confirm record counts, remaining missing values and duplicate rows.

In [21]:
final_quality = []

for name, df in datasets.items():
    final_quality.append({
        "Dataset": name,
        "Rows": len(df),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })

final_quality_df = pd.DataFrame(final_quality)

display(final_quality_df)

,Dataset,Rows,Missing Values,Duplicate Rows
0,product_category_name_translation,71,0,0
1,olist_order_reviews_dataset,99224,0,0
2,olist_order_payments_dataset,103886,0,0
3,olist_products_dataset,32951,1838,0
4,olist_geolocation_dataset,738332,0,0
5,olist_customers_dataset,99441,0,0
6,olist_sellers_dataset,3095,0,0
7,olist_orders_dataset,99441,4908,0
8,olist_order_items_dataset,112650,0,0


### **Create Cleaned Dataset Backup**

Package all nine cleaned CSV files into one ZIP archive for future use.

In [25]:
import shutil

zip_path = shutil.make_archive(
    '/content/cleaned_datasets_backup',
    'zip',
    processed_path
)

print("ZIP backup created:", zip_path)

ZIP backup created: /content/cleaned_datasets_backup.zip


In [26]:
from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Data Cleaning Summary**

- Removed 261,831 exact duplicate records from the geolocation dataset.
- Replaced missing review comments with empty strings.
- Replaced missing product categories with `Unknown`.
- Preserved products with missing descriptive attributes and physical measurements.
- Retained orders with missing timestamps without inserting estimated dates.
- Converted order timestamp columns to datetime format.
- Saved all nine updated datasets in the processed directory.
- Performed final data quality validation.
- Confirmed zero exact duplicate rows across all nine datasets.